# **Hypothesis Testing**
### หาเหตุผลรองรับการเลือก ว่า v4.5 หรือ v4.6
#### ทดสอบด้วย
- **Bootstrap Resampling Hypothesis Testing**

    และ
-  **Two-Proportion Z-Test Hypothesis Testing**

> 13 April 2026

---

# **Bootstrap Resampling Hypothesis Testing**

#### ไฟล์ที่เกี่ยวข้อง (ผลจาการรัน pipline v4 ipynb)
- evauation-for-hypothesis_testing/debug_all_y_true.txt
- evauation-for-hypothesis_testing/debug_all_y_pred_45.txt
- evauation-for-hypothesis_testing/debug_all_y_pred_46.txt

In [1]:
import numpy as np
from sklearn.metrics import f1_score
from tqdm import tqdm

def load_data(file_path):
    """ฟังก์ชันสำหรับอ่านไฟล์ text และทำความสะอาดข้อมูล (ลบช่องว่าง)"""
    with open(file_path, "r", encoding="utf-8") as f:
        # อ่านทีละบรรทัด ลบ whitespace ซ้ายขวา และข้ามบรรทัดว่าง
        return np.array([line.strip() for line in f.readlines() if line.strip()])

# =========================================================
# 1. โหลดข้อมูลจากไฟล์ทั้ง 3 อัน
# =========================================================
print("Loading data...")
try:
    y_true = load_data("evauation-for-hypothesis_testing/debug_all_y_true.txt")
    y_pred_45 = load_data("evauation-for-hypothesis_testing/debug_all_y_pred_45.txt")
    y_pred_46 = load_data("evauation-for-hypothesis_testing/debug_all_y_pred_46.txt") 
except FileNotFoundError as e:
    print(f"Error: หาไฟล์ไม่เจอ กรุณาตรวจสอบว่าไฟล์อยู่ในโฟลเดอร์เดียวกัน - {e}")
    exit()

n_samples = len(y_true)
print(f"Loaded {n_samples} samples successfully.\n")

# ตรวจสอบความยาวให้ชัวร์ว่าเท่ากันหมด
if not (len(y_true) == len(y_pred_45) == len(y_pred_46)):
    print("Error: จำนวนข้อมูลในไฟล์ไม่เท่ากัน!")
    exit()

# =========================================================
# 2. คำนวณค่า Observed Difference (F1 ของ v4.5 ลบด้วย v4.6)
# =========================================================
# ใช้ average='macro' ตามที่อยู่ในตาราง Evaluation ของคุณ
obs_f1_45 = f1_score(y_true, y_pred_45, average="macro", zero_division=0)
obs_f1_46 = f1_score(y_true, y_pred_46, average="macro", zero_division=0)
observed_diff = obs_f1_45 - obs_f1_46

print("========== OBSERVED METRICS ==========")
print(f"F1 v4.5       : {obs_f1_45:.4f}")
print(f"F1 v4.6       : {obs_f1_46:.4f}")
print(f"Observed Diff : {observed_diff:.4f} (v4.5 - v4.6)\n")

# =========================================================
# 3. เริ่มกระบวนการ Bootstrap Resampling
# =========================================================
n_boot = 1000
rng = np.random.default_rng(42)  # ล็อก seed ให้ผลลัพธ์ออกมาระบุได้ตรงกันทุกครั้งที่รัน
diffs = []

print(f"========== BOOTSTRAP TESTING ({n_boot} iterations) ==========")
for _ in tqdm(range(n_boot), desc="Resampling"):
    # สุ่ม index ขึ้นมาใหม่จำนวน n_samples ตัว (แบบใส่คืน/replace=True)
    idx = rng.choice(n_samples, size=n_samples, replace=True)
    
    # ดึงข้อมูลตาม index ที่สุ่มได้
    yt = y_true[idx]
    ya = y_pred_45[idx]
    yb = y_pred_46[idx]
    
    # คำนวณ F1 ของรอบนี้และหาค่าความต่าง
    b_f1_45 = f1_score(yt, ya, average="macro", zero_division=0)
    b_f1_46 = f1_score(yt, yb, average="macro", zero_division=0)
    diffs.append(b_f1_45 - b_f1_46)

diffs = np.array(diffs)

# =========================================================
# 4. คำนวณช่วงความเชื่อมั่น (95% CI) และ P-Value ที่ถูกต้อง
# =========================================================
ci_lower = np.percentile(diffs, 2.5)
ci_upper = np.percentile(diffs, 97.5)

# การคำนวณ P-Value แบบ Two-sided สำหรับ Bootstrap (วิธีที่ถูกต้อง)
# ดูว่าสัดส่วนที่ผลต่างข้ามฝั่งไปเป็นศูนย์หรือติดลบมีเท่าไหร่ แล้วคูณ 2 (Two-tailed test)
p_value = 2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0))

print("\n========== HYPOTHESIS TESTING RESULT ==========")
print(f"95% CI        : [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"P-value       : {p_value:.4f}")

if p_value < 0.05:
    print("RESULT: Significant difference (ความแม่นยำแตกต่างกันอย่างมีนัยสำคัญ)")
else:
    print("RESULT: No significant difference (ความแม่นยำไม่แตกต่างกันอย่างมีนัยสำคัญ)")

Loading data...
Loaded 1979 samples successfully.

========== OBSERVED METRICS ==========
F1 v4.5       : 0.4714
F1 v4.6       : 0.4601
Observed Diff : 0.0113 (v4.5 - v4.6)

========== BOOTSTRAP TESTING (1000 iterations) ==========


Resampling: 100%|██████████| 1000/1000 [00:02<00:00, 350.18it/s]


========== HYPOTHESIS TESTING RESULT ==========
95% CI        : [-0.0091, 0.0307]
P-value       : 0.2820
RESULT: No significant difference (ความแม่นยำไม่แตกต่างกันอย่างมีนัยสำคัญ)


---

#### **F1 v4.5:** 0.4714 | **F1 v4.6:** 0.4601 | **Observed Diff:** 0.0113

- **ความหมาย:** จากข้อมูลจริงทั้งหมด 1,979 เคส โมเดล v4.5 ทำคะแนน Macro F1 ได้สูงกว่า v4.6 อยู่เล็กน้อย คือต่างกันแค่ 0.0113 (หรือประมาณ 1.13%)
- **จุดสังเกต:** แม้ว่า Accuracy ของ v4.5 จะดูทิ้งห่าง (70% vs 60%) แต่พอเราใช้ F1-Score แบบ Macro ที่ให้น้ำหนักทุกคลาสอย่างเท่าเทียมกัน (ไม่โดนหลอกโดยคลาสใหญ่) เราจะเห็นความจริงว่า "ประสิทธิภาพที่แท้จริงของทั้งสองโมเดล สูสีกันมาก" ### 2. ผลการทดสอบสมมติฐาน (Hypothesis Testing Result)


    **95% CI (Confidence Interval): [-0.0091, 0.0307]**

- **ความหมาย:** เมื่อเราจำลองการสุ่มข้อมูลใหม่ 1,000 รอบ (Bootstrap) เราพบว่าความต่างของคะแนน (v4.5 ลบ v4.6) จะแกว่งตัวอยู่ในช่วงติดลบ 0.0091 ไปจนถึงบวก 0.0307

- **จุดชี้เป็นชี้ตาย:** สังเกตว่าช่วงความเชื่อมั่นนี้ "ครอบคลุมเลข 0 (คร่อมศูนย์)" ครับ การที่ช่วงมันคร่อมศูนย์ (จากติดลบไปเป็นบวก) แปลว่าในบางครั้งที่สุ่มข้อมูล โมเดล v4.6 ก็สามารถพลิกกลับมาชนะโมเดล v4.5 ได้ (ผลต่างติดลบ) นี่คือสัญญาณทางสถิติที่บอกชัดเจนว่า โมเดลสองตัวนี้เก่งพอๆ กัน

    **P-value: 0.2820**

- **ความหมาย:** ค่า P-value คือ 0.2820 แปลว่า "มีโอกาสถึง 28.2% ที่ความต่าง 1.13% ที่เราเห็นตอนแรก จะเกิดขึ้นจากความบังเอิญของข้อมูล ไม่ใช่เพราะโมเดล v4.5 เก่งกว่าจริงๆ"

- **การตัดสินใจ:** ในทางสถิติ เราใช้เกณฑ์ความคลาดเคลื่อนที่ยอมรับได้คือไม่เกิน 5% (P-value < 0.05) แต่ 28.2% นั้นสูงกว่าเกณฑ์มาก เราจึง "ยอมรับสมมติฐานหลัก (Accept H0)"

## บทสรุป (Conclusion) ของ Bootstrap Hypothesis Testing

จากการประเมินผลด้วย F1-Score (Macro Average) เพื่อลดอคติจากปัญหา Data Imbalance พบว่าโมเดล v4.5 มีคะแนนเฉลี่ยสูงกว่า v4.6 เล็กน้อย (0.4714 เทียบกับ 0.4601) อย่างไรก็ตาม เมื่อทำการทดสอบนัยสำคัญทางสถิติด้วยวิธี Bootstrap Resampling (1,000 รอบ) พบว่า ค่า P-value เท่ากับ 0.2820 (p > 0.05) และช่วงความเชื่อมั่น 95% ครอบคลุมค่าศูนย์ [-0.0091, 0.0307]

จึงสรุปได้ว่า ประสิทธิภาพในการจำแนกพื้นผิวรอยผุของโมเดล v4.5 และ v4.6 ไม่แตกต่างกันอย่างมีนัยสำคัญทางสถิติ แม้ว่า v4.6 จะมีตรรกะการแบ่งโซนที่ช่วยแก้ปัญหารอยผุมุมโค้งได้ดี (ทำให้ Recall คลาส Occlusal เพิ่มขึ้น) แต่ผลกระทบข้างเคียงที่ทำให้คลาส Distal คลาดเคลื่อน ก็หักล้างกันจนทำให้ประสิทธิภาพโดยรวมกลับมาสูสีกับ v4.5 ในที่สุด

---
---

# **Two-Proportion Z-Test Hypothesis Testing**

```
========== FINAL EVALUATION [v4.5] ==========
Output Dir    : PCA_Output_v4.5
Total Samples : 1979
Accuracy      : 0.7054
Precision     : 0.5463
Recall        : 0.4645
F1 Score      : 0.4714

========== CONFUSION MATRIX [v4.5] ==========
          Occlusal  Mesial  Distal  Other
Occlusal        78     121     152     37
Mesial           5     555      80     30
Distal          31      80     763     47
Other            0       0       0      0

========== CLASSIFICATION REPORT [v4.5] ==========
              precision    recall  f1-score   support
    Occlusal       0.68      0.20      0.31       388
      Mesial       0.73      0.83      0.78       670
      Distal       0.77      0.83      0.80       921
       Other       0.00      0.00      0.00         0
    accuracy                           0.71      1979
   macro avg       0.55      0.46      0.47      1979
weighted avg       0.74      0.71      0.70      1979
```

```
========== FINAL EVALUATION [v4.6] ==========
Output Dir    : PCA_Output_v4.6
Total Samples : 1979
Accuracy      : 0.6028
Precision     : 0.5131
Recall        : 0.4634
F1 Score      : 0.4601

========== CONFUSION MATRIX [v4.6] ==========
          Occlusal  Mesial  Distal  Other
Occlusal       269      43      39     37
Mesial         211     386      43     30
Distal         307      29     538     47
Other            0       0       0      0

========== CLASSIFICATION REPORT [v4.6] ==========
              precision    recall  f1-score   support
    Occlusal       0.34      0.69      0.46       388
      Mesial       0.84      0.58      0.68       670
      Distal       0.87      0.58      0.70       921
       Other       0.00      0.00      0.00         0
    accuracy                           0.60      1979
   macro avg       0.51      0.46      0.46      1979
weighted avg       0.76      0.60      0.65      1979
```

In [1]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# =========================================================
# 1. ข้อมูลจากผลประเมิน (Evaluation Results)
# =========================================================
n_total = 1979

# นับจำนวนที่ทายถูกจากแนวทแยงของ Confusion Matrix
# v4.5: Occlusal(78) + Mesial(555) + Distal(763)
correct_45 = 78 + 555 + 763   # รวมได้ 1396 (Accuracy = 0.7054)

# v4.6: Occlusal(269) + Mesial(386) + Distal(538)
correct_46 = 269 + 386 + 538  # รวมได้ 1193 (Accuracy = 0.6028)

# =========================================================
# 2. ทำ Two-Proportion Z-Test
# =========================================================
# สมมติฐาน (Hypothesis)
# H0 (Null Hypothesis) : Accuracy ของทั้งสองเวอร์ชันไม่แตกต่างกัน
# H1 (Alt Hypothesis)  : Accuracy ของทั้งสองเวอร์ชันแตกต่างกัน

counts = np.array([correct_45, correct_46])
nobs = np.array([n_total, n_total])

# คำนวณค่าสถิติ Z-Statistic และ P-value
stat, p_value = proportions_ztest(counts, nobs)

# =========================================================
# 3. แสดงผลลัพธ์
# =========================================================
print("========== HYPOTHESIS TESTING (Z-TEST FOR ACCURACY) ==========")
print(f"v4.5 Accuracy : {correct_45/n_total:.4f} ({correct_45}/{n_total})")
print(f"v4.6 Accuracy : {correct_46/n_total:.4f} ({correct_46}/{n_total})")
print(f"Z-Statistic   : {stat:.4f}")
print(f"P-Value       : {p_value:.4e}")
print("==============================================================")

if p_value < 0.05:
    print("RESULT: ปฏิเสธ H0 (Significant difference)")
    print("สรุป: ความแม่นยำ (Accuracy) ของ v4.5 แตกต่างจาก v4.6 อย่างมีนัยสำคัญทางสถิติที่ระดับความเชื่อมั่น 95%")
else:
    print("RESULT: ยอมรับ H0 (No significant difference)")
    print("สรุป: ความแม่นยำ (Accuracy) ของทั้งสองโมเดลไม่แตกต่างกันอย่างมีนัยสำคัญ")

========== HYPOTHESIS TESTING (Z-TEST FOR ACCURACY) ==========
v4.5 Accuracy : 0.7054 (1396/1979)
v4.6 Accuracy : 0.6028 (1193/1979)
Z-Statistic   : 6.7837
P-Value       : 1.1714e-11
RESULT: ปฏิเสธ H0 (Significant difference)
สรุป: ความแม่นยำ (Accuracy) ของ v4.5 แตกต่างจาก v4.6 อย่างมีนัยสำคัญทางสถิติที่ระดับความเชื่อมั่น 95%


---

### โมเดล v4.5 สามารถทำนายรอยผุได้แม่นยำกว่า v4.6 อย่างแน่นอน ไม่ใช่เรื่องบังเอิญ
**1. มองที่ส่วนต่างของ Accuracy (1396 vs 1193)**
- ความหมาย: จากฟันผุทั้งหมด 1,979 เคส โมเดล v4.5 ทายถูก 1,396 เคส ส่วน v4.6 ทายถูกเพียง 1,193 เคส
- คำอธิบาย: โมเดล v4.5 ทายถูกมากกว่าถึง 203 เคส คิดเป็นส่วนต่างความแม่นยำประมาณ 10% ซึ่งในทาง Machine Learning ถือว่าเป็นช่องว่างที่กว้างมากครับ

**2. ค่า Z-Statistic : 6.7837 (คะแนนความห่างเหิน)**
- ความหมาย: ค่า Z คือตัวชี้วัดทางสถิติที่บอกว่า ความแม่นยำของโมเดลทั้งสองตัวนี้ "ห่างกัน" แค่ไหนในรูปแบบของกราฟระฆังคว่ำ
- คำอธิบาย: ตามหลักสถิติมาตรฐาน ถ้าค่า Z เกิน 1.96 (หรือต่ำกว่า -1.96) เราก็จะถือว่ามันแตกต่างกันแล้วครับ แต่นี่ค่า Z ทะลุไปถึง 6.78
    
    แปลว่ากราฟความแม่นยำของสองโมเดลนี้ฉีกห่างออกจากกันแบบสุดกู่เลยครับ

**3. ค่า P-Value : 1.1714e-11 (โอกาสที่จะฟลุค)**
- ความหมาย: 1.1714e-11 เป็นสัญกรณ์วิทยาศาสตร์ (Scientific Notation) ถ้าเขียนเป็นตัวเลขปกติมันคือ 0.000000000011714 (มีศูนย์ 10 ตัวอยู่หน้าเลข 1)

- คำอธิบาย: เกณฑ์มาตรฐานที่เรายอมรับคือ 0.05 (โอกาสฟลุค 5%) แต่ตัวเลขที่ได้มานี้คือ โอกาสเข้าใกล้ศูนย์เป๊ะๆ แปลว่า โอกาสที่โมเดล v4.5 จะบังเอิญได้คะแนนเยอะกว่า v4.6 เพราะโชคช่วยนั้น "แทบเป็นไปไม่ได้เลย" ชัยชนะครั้งนี้มาจากความสามารถของตัวอัลกอริทึม v4.5 ล้วนๆ ครับ

## บทสรุป (Conclusion) ของ Two-Proportion Z-Test Hypothesis Testing

ผลจากการทดสอบสมมติฐานด้วย Two-Proportion Z-Test เพื่อเปรียบเทียบความสามารถในการทายถูกชิ้นต่อชิ้น (Accuracy) พบว่า โมเดล v4.5 มีความแม่นยำอยู่ที่ 70.54% ในขณะที่ v4.6 อยู่ที่ 60.28%

เมื่อวิเคราะห์ทางสถิติ เราได้ค่า P-Value ที่ 1.17e-11 ซึ่งน้อยกว่า 0.05 มากๆ และมีค่า Z-statistic สูงถึง 6.78 จึงสรุปได้อย่างมั่นใจ 100% ว่า เราปฏิเสธสมมติฐานหลัก (Reject H0) และยืนยันได้ว่าโมเดล v4.5 มีความแม่นยำในภาพรวมสูงกว่า v4.6 อย่างมีนัยสำคัญทางสถิติครับ

สาเหตุหลักมาจากชุดข้อมูลของเรามีรอยผุด้านข้าง (Mesial/Distal) เป็นจำนวนมาก ซึ่งอัลกอริทึมของ v4.5 มีพื้นที่ดักจับด้านข้างกว้างกว่า ทำให้สามารถเก็บคะแนนจากคลาสส่วนใหญ่เหล่านี้ไปได้เกือบทั้งหมด

---

# อธิบายใส่รายงาน

### **บทสรุปและวิเคราะห์ผลการเปรียบเทียบอัลกอริทึม (v4.5 vs v4.6)**

ในการพัฒนาระบบระบุตำแหน่งรอยผุบนพื้นผิวฟัน (Surface Classification) คณะผู้จัดทำได้เปรียบเทียบอัลกอริทึมการแบ่งโซน 2 รูปแบบ ได้แก่ **v4.5 (แบ่งแถบแนวตั้ง X-thirds)** และ **v4.6 (แบ่งเส้นทแยงมุม Diagonal-from-Centroid)** โดยทำการประเมินผลบนชุดข้อมูลจำนวน 1,979 ตัวอย่าง และทดสอบสมมติฐานทางสถิติ (Hypothesis Testing) ใน 2 มิติ ดังนี้:

**1. การทดสอบมิติด้านความแม่นยำรวม (Accuracy) ด้วย Two-Proportion Z-Test**
* **ผลการทดสอบ:** โมเดล v4.5 มีความแม่นยำรวมอยู่ที่ 70.54% ในขณะที่ v4.6 อยู่ที่ 60.28%
* **นัยสำคัญทางสถิติ:** จากการทดสอบพบค่า P-value = $1.17 \times 10^{-11}$ ซึ่งน้อยกว่า 0.05 มาก และมีค่า Z-Statistic สูงถึง 6.78
* **ข้อสรุป:** ปฏิเสธสมมติฐานหลัก (Reject H0) ยืนยันได้ว่าโมเดล v4.5 มีความสามารถในการทำนายผลภาพรวมได้แม่นยำกว่า v4.6 อย่างมีนัยสำคัญทางสถิติ 

**2. การทดสอบมิติด้านความสมดุลของคลาส (Macro F1-Score) ด้วย Bootstrap Resampling**
เนื่องจากชุดข้อมูลมีสภาวะขาดความสมดุล (Data Imbalance) โดยมีคลาส Mesial และ Distal รวมกันถึง 80% คณะผู้จัดทำจึงประเมินด้วย Macro F1-Score เพื่อความยุติธรรมต่อทุกคลาส
* **ผลการทดสอบ:** โมเดล v4.5 มีค่า Macro F1 เท่ากับ 0.4714 ส่วน v4.6 เท่ากับ 0.4601 (ผลต่างเพียง 0.0113)
* **นัยสำคัญทางสถิติ:** เมื่อจำลองข้อมูลซ้ำด้วยเทคนิค Bootstrap จำนวน 1,000 รอบ พบว่าช่วงความเชื่อมั่น 95% (CI) อยู่ที่ [-0.0091, 0.0307] ซึ่งครอบคลุมค่าศูนย์ และมีค่า P-value = 0.282 (p > 0.05)
* **ข้อสรุป:** ยอมรับสมมติฐานหลัก (Accept H0) ประสิทธิภาพในการทำนายรอยผุแบบรายคลาสของทั้งสองโมเดล ไม่แตกต่างกันอย่างมีนัยสำคัญทางสถิติ

**3. การวิเคราะห์ปัญหาคลาดเคลื่อน (Error Analysis & Trade-off)**
เมื่อวิเคราะห์เชิงลึกจาก Confusion Matrix พบว่า อัลกอริทึม v4.6 สามารถแก้ปัญหาการทำนายรอยผุบริเวณมุมโค้งของคลาส Occlusal ได้ดีเยี่ยมตามสมมติฐาน (Recall เพิ่มขึ้นอย่างก้าวกระโดดจาก 0.20 เป็น 0.69) แต่ต้องแลกมากับปัญหาผลกระทบข้างเคียง (Trade-off) ในคลาส Distal 
สาเหตุหลักเกิดจาก **"ปัญหาความสูงของ Bounding Box"** เนื่องจากภาพที่ป้อนเข้าสู่ระบบยังรวมส่วนของรากฟันเอาไว้ ทำให้กรอบฟันเป็นสี่เหลี่ยมทรงสูง เมื่อ v4.6 ตีเส้นทแยงมุม พื้นที่สามเหลี่ยมด้านบน (Occlusal) จึงถูกยืดออกกว้างเกินไปและไปแย่งพื้นที่สามเหลี่ยมด้านข้าง ส่งผลให้รอยผุด้าน Distal ถูกทำนายผิดเป็น Occlusal จำนวนมาก

**สรุปการตัดสินใจ (The Verdict)**
จากผลการวิเคราะห์เชิงสถิติและกายวิภาค คณะผู้จัดทำมีมติ **"เลือกใช้อัลกอริทึม v4.5 เป็นโมเดลหลักในเฟสนี้"** เนื่องจากมีความแม่นยำสูงกว่าอย่างมีนัยสำคัญ สามารถนำไปใช้งานได้จริงอย่างมีเสถียรภาพ และสอดคล้องกับสภาพของชุดข้อมูลปัจจุบัน 

**ข้อเสนอแนะสำหรับการพัฒนาต่อ (Future Work):**
สำหรับอัลกอริทึม v4.6 ถือเป็นแนวคิดที่มีความถูกต้องในเชิงคณิตศาสตร์เรขาคณิต แต่อัลกอริทึมนี้จะแสดงประสิทธิภาพสูงสุดได้ก็ต่อเมื่อมีการทำ **Crown-Root Separation (การตัดรากฟันออกก่อนการแบ่งโซน)** ซึ่งจะช่วยปรับสัดส่วนกรอบฟันให้สมดุล และกำจัดผลกระทบข้างเคียงที่เกิดขึ้นกับคลาส Distal ได้อย่างสมบูรณ์ จึงเสนอให้นำ v4.6 ไปใช้ในกระบวนการพัฒนาเฟสถัดไป